# PySpark Challenges

## Prepare environment

In [1]:
# Importing libraries
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession, Window, functions as F

In [2]:
# Connecto to spark
spark = SparkSession.builder.getOrCreate()

## Provide the data

In [3]:
# Set the number of rows
num_rows = 10_000_000

# Generate random TransactionID as sequential integers
transaction_ids = np.arange(1, num_rows + 1)

# Generate random UserID (assume 1 million unique users)
user_ids = np.random.choice([f"U{i:06}" for i in range(1, 1_000_001)], size=num_rows)

# Generate random Amount between 1 and 1000
amounts = np.random.uniform(1, 1000, size=num_rows)

# Generate random TransactionDate between a range of dates
transaction_dates = pd.date_range(start="2022-01-01", end="2023-12-31", periods=num_rows)

# Create the DataFrame
data = {
    "TransactionID": transaction_ids,
    "UserID": user_ids,
    "Amount": amounts,
    "TransactionDate": transaction_dates
}

df = pd.DataFrame(data)

# Save to CSV file
df.to_csv('data/large_transactions_data.csv', index=False)
print("Data generated and saved as 'large_transactions_data.csv'.")
df.head()

Data generated and saved as 'large_transactions_data.csv'.


,TransactionID,UserID,Amount,TransactionDate
0,1,U723998,602.904129,2022-01-01 00:00:00.000000000
1,2,U081247,307.179662,2022-01-01 00:00:06.298560629
2,3,U231696,566.529045,2022-01-01 00:00:12.597121259
3,4,U902610,178.492793,2022-01-01 00:00:18.895681889
4,5,U413666,344.430617,2022-01-01 00:00:25.194242519


In [4]:
!dir data\ /b

large_table.parquet
large_transactions_data.csv
orders.csv
sales_data.csv
users.csv


## Challenges

### Challenge 1: Data Transformation
You are given a CSV file containing sales data with the following columns: OrderID, ProductID, Quantity, PricePerUnit, OrderDate, and CustomerID. The task is to:

1. Load the data from the CSV into a PySpark DataFrame.
2. Calculate the total sales amount for each order by multiplying Quantity by PricePerUnit.
3. Filter out the orders where the total sales amount is less than $500.
4. Group the data by CustomerID and calculate the total amount spent by each customer.
5. Sort the customers by the total amount spent in descending order.

**Expected Outcome:** The candidate should return a DataFrame that lists customers and their total spending, sorted from highest to lowest.

In [5]:
# (1) Load the data from the CSV into a PySpark DataFrame.
df_sales = spark.read.csv('data/sales_data.csv', header=True, inferSchema=True)
df_sales.printSchema()
df_sales.show()

root
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- PricePerUnit: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)

+-------+---------+--------+------------+----------+----------+
|OrderID|ProductID|Quantity|PricePerUnit| OrderDate|CustomerID|
+-------+---------+--------+------------+----------+----------+
|   1001|     2001|       3|       150.0|2023-10-01|      C001|
|   1002|     2003|       5|        90.0|2023-10-02|      C002|
|   1003|     2001|       2|       150.0|2023-10-02|      C001|
|   1004|     2004|       1|       600.0|2023-10-03|      C003|
+-------+---------+--------+------------+----------+----------+



In [6]:
# (2) Calculate the total sales amount for each order by 
#     multiplying Quantity by PricePerUnit.
df_sales.withColumn('TotalSales', df_sales.Quantity * df_sales.PricePerUnit).show()

+-------+---------+--------+------------+----------+----------+----------+
|OrderID|ProductID|Quantity|PricePerUnit| OrderDate|CustomerID|TotalSales|
+-------+---------+--------+------------+----------+----------+----------+
|   1001|     2001|       3|       150.0|2023-10-01|      C001|     450.0|
|   1002|     2003|       5|        90.0|2023-10-02|      C002|     450.0|
|   1003|     2001|       2|       150.0|2023-10-02|      C001|     300.0|
|   1004|     2004|       1|       600.0|2023-10-03|      C003|     600.0|
+-------+---------+--------+------------+----------+----------+----------+



In [7]:
w = (Window.partitionBy('OrderID')
           .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))

df_sales = df_sales.withColumn('Total', df_sales.Quantity * df_sales.PricePerUnit)
df_sales = df_sales.withColumn('TotalSales', 
                               F.sum(df_sales.Total).over(w))
df_sales.show()

+-------+---------+--------+------------+----------+----------+-----+----------+
|OrderID|ProductID|Quantity|PricePerUnit| OrderDate|CustomerID|Total|TotalSales|
+-------+---------+--------+------------+----------+----------+-----+----------+
|   1001|     2001|       3|       150.0|2023-10-01|      C001|450.0|     450.0|
|   1002|     2003|       5|        90.0|2023-10-02|      C002|450.0|     450.0|
|   1003|     2001|       2|       150.0|2023-10-02|      C001|300.0|     300.0|
|   1004|     2004|       1|       600.0|2023-10-03|      C003|600.0|     600.0|
+-------+---------+--------+------------+----------+----------+-----+----------+



In [8]:
# (3) Filter out the orders where the total sales amount is less than $500.
df_sales.filter(df_sales.TotalSales >= 500).show()

+-------+---------+--------+------------+----------+----------+-----+----------+
|OrderID|ProductID|Quantity|PricePerUnit| OrderDate|CustomerID|Total|TotalSales|
+-------+---------+--------+------------+----------+----------+-----+----------+
|   1004|     2004|       1|       600.0|2023-10-03|      C003|600.0|     600.0|
+-------+---------+--------+------------+----------+----------+-----+----------+



In [9]:
# (4) Group the data by CustomerID and calculate 
#     the total amount spent by each customer.
df_sales.groupby('CustomerID').sum('TotalSales').show()

+----------+---------------+
|CustomerID|sum(TotalSales)|
+----------+---------------+
|      C003|          600.0|
|      C001|          750.0|
|      C002|          450.0|
+----------+---------------+



In [10]:
# (5) Sort the customers by the total amount spent in descending order.
# Expected Outcome: 
# The candidate should return a DataFrame that lists customers and their 
# total spending, sorted from highest to lowest.
(df_sales.groupby('CustomerID')
         .agg(F.sum('TotalSales').alias('TotalSales'))
         .orderBy(F.col('TotalSales').desc())).show()

+----------+----------+
|CustomerID|TotalSales|
+----------+----------+
|      C001|     750.0|
|      C003|     600.0|
|      C002|     450.0|
+----------+----------+



### Challenge 2: Join and Aggregation
You are provided with two datasets in CSV format:

- users.csv with columns: UserID, Name, and Location
- orders.csv with columns: OrderID, UserID, Amount, and OrderDate

The tasks are:

1. Load both datasets into PySpark DataFrames.
2. Join the two DataFrames on UserID to combine the user and order information.
3. Find the total amount spent by each user and show only the users who have placed more than 2 orders.
4. Group the results by Location and calculate the total amount spent in each location.

**Expected Outcome:** The candidate should return a DataFrame that shows the total amount spent by users in different locations, only for users with more than 2 orders.

In [11]:
# (1) Load both datasets into PySpark DataFrames.
df_user = spark.read.csv('data/users.csv', header=True, inferSchema=True)
df_user.printSchema()
df_user.show()

root
 |-- UserID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Location: string (nullable = true)

+------+-----+--------+
|UserID| Name|Location|
+------+-----+--------+
|  U001|Alice|New York|
|  U002|  Bob|  London|
|  U003|Carol|   Tokyo|
+------+-----+--------+



In [12]:
df_order = spark.read.csv('data/orders.csv', header=True, inferSchema=True)
df_order.printSchema()
df_order.show()

root
 |-- OrderID: integer (nullable = true)
 |-- UserID: string (nullable = true)
 |-- Amount: integer (nullable = true)
 |-- OrderDate: date (nullable = true)

+-------+------+------+----------+
|OrderID|UserID|Amount| OrderDate|
+-------+------+------+----------+
|   1001|  U001|   500|2023-10-01|
|   1002|  U002|   200|2023-10-01|
|   1003|  U001|   300|2023-10-02|
|   1004|  U001|   150|2023-10-03|
|   1005|  U003|   600|2023-10-03|
+-------+------+------+----------+



In [13]:
# (2) Join the two DataFrames on UserID to combine the 
#     user and order information.
df_shipment = df_user.join(df_order, on='UserID', how='left')
df_shipment.show()

+------+-----+--------+-------+------+----------+
|UserID| Name|Location|OrderID|Amount| OrderDate|
+------+-----+--------+-------+------+----------+
|  U001|Alice|New York|   1004|   150|2023-10-03|
|  U001|Alice|New York|   1003|   300|2023-10-02|
|  U001|Alice|New York|   1001|   500|2023-10-01|
|  U002|  Bob|  London|   1002|   200|2023-10-01|
|  U003|Carol|   Tokyo|   1005|   600|2023-10-03|
+------+-----+--------+-------+------+----------+



In [14]:
# (3) Find the total amount spent by each user and show 
#     only the users who have placed more than 2 orders.
(df_shipment.groupBy('UserID', 'Name')
            .agg(F.sum('Amount').alias('TotalSpent'),
                 F.count('OrderID').alias('TotalOrders'))
            .where(F.col('TotalOrders')>2)
            .select('UserID', 'Name', 'TotalSpent')).show()

+------+-----+----------+
|UserID| Name|TotalSpent|
+------+-----+----------+
|  U001|Alice|       950|
+------+-----+----------+



In [15]:
# (4) Group the results by Location and calculate the 
#     total amount spent in each location.
(df_shipment.groupBy('Location')
            .agg(F.sum('Amount').alias('TotalSpent'))).show()

+--------+----------+
|Location|TotalSpent|
+--------+----------+
|  London|       200|
|   Tokyo|       600|
|New York|       950|
+--------+----------+



In [16]:
# Expected Outcome: 
# The candidate should return a DataFrame that shows the total 
# amount spent by users in different locations, only for users 
# with more than 2 orders.
(
    df_shipment.groupBy('UserID', 'Name', 'Location')
               .agg(F.sum('Amount').alias('TotalSpent'),
                    F.count('OrderID').alias('TotalOrders'))
               .where(F.col('TotalOrders')>2)
               .select('UserID', 'Name', 'Location', 'TotalSpent')
).show()

+------+-----+--------+----------+
|UserID| Name|Location|TotalSpent|
+------+-----+--------+----------+
|  U001|Alice|New York|       950|
+------+-----+--------+----------+



### Challenge 3: Handling Large Data and Optimization
You are given a dataset with 10 million rows, containing the following columns: TransactionID, UserID, Amount, and TransactionDate. Assume this dataset is stored in a CSV file and the task is to:

1. Load the dataset into a PySpark DataFrame.
2. Partition the data by TransactionDate and UserID to optimize further operations.
3. Calculate the average transaction amount for each user.
4. Identify the top 10 users who have the highest average transaction amounts.

This challenge tests their ability to work with large datasets efficiently, optimize performance, and handle memory constraints.

In [17]:
# (1) Load the dataset into a PySpark DataFrame.
df_sales = spark.read.csv('data/large_transactions_data.csv', 
                          header=True, inferSchema=True)
df_sales.printSchema()
df_sales.show(10)

root
 |-- TransactionID: integer (nullable = true)
 |-- UserID: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- TransactionDate: timestamp (nullable = true)

+-------------+-------+------------------+--------------------+
|TransactionID| UserID|            Amount|     TransactionDate|
+-------------+-------+------------------+--------------------+
|            1|U020119| 395.6050410767464| 2022-01-01 00:00:00|
|            2|U568945|151.28154953730282|2022-01-01 00:00:...|
|            3|U742006|  934.510068072298|2022-01-01 00:00:...|
|            4|U257075| 811.4821203995795|2022-01-01 00:00:...|
|            5|U208251| 502.8912068788912|2022-01-01 00:00:...|
|            6|U309923| 649.7561867095405|2022-01-01 00:00:...|
|            7|U302554| 963.3579431229631|2022-01-01 00:00:...|
|            8|U764232|509.79658079647385|2022-01-01 00:00:...|
|            9|U957384|  59.2937681006926|2022-01-01 00:00:...|
|           10|U070709| 774.9236679088821|2022-01-01 0

In [18]:
# (2) Partition the data by TransactionDate and UserID 
#     to optimize further operations.
print('Current Number of partitions:', df_sales.rdd.getNumPartitions())
df_sales = (df_sales.withColumn('Date', F.to_date(df_sales.TransactionDate))
                    .withColumn('Year', F.year(df_sales.TransactionDate))
                    .withColumn('Month', F.month(df_sales.TransactionDate))
                    .withColumn('Day', F.day(df_sales.TransactionDate))
                    .repartition('UserID', 'Year', 'Month', 'Day'))
df_sales.show(5)
print('After changes, Number of partitions:', df_sales.rdd.getNumPartitions())

Current Number of partitions: 16
+-------------+-------+------------------+--------------------+----------+----+-----+---+
|TransactionID| UserID|            Amount|     TransactionDate|      Date|Year|Month|Day|
+-------------+-------+------------------+--------------------+----------+----+-----+---+
|          110|U433278|143.39178068518103|2022-01-01 00:11:...|2022-01-01|2022|    1|  1|
|          205|U266497| 271.7615062872951|2022-01-01 00:21:...|2022-01-01|2022|    1|  1|
|          247|U823223| 927.9655279363044|2022-01-01 00:25:...|2022-01-01|2022|    1|  1|
|          259|U315626| 925.8221189323317|2022-01-01 00:27:...|2022-01-01|2022|    1|  1|
|          389|U050530| 286.2035988243543|2022-01-01 00:40:...|2022-01-01|2022|    1|  1|
+-------------+-------+------------------+--------------------+----------+----+-----+---+
only showing top 5 rows

After changes, Number of partitions: 17


In [19]:
# df_sales.write\
#         .mode('overwrite')\
#         .partitionBy('UserID', 'Year', 'Month', 'Day')\
#         .saveAsTable('large_table')

In [20]:
df_sales.createOrReplaceTempView("large_table")

In [21]:
spark.catalog.listTables()

[Table(name='large_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [22]:
df_sales.write.parquet('data/large_table.parquet', mode="overwrite")

In [23]:
# (3) Calculate the average transaction amount for each user.
(df_sales.groupBy('UserID')
         .agg(F.avg('Amount').alias('AverageAmount'))).show(5)

+-------+-----------------+
| UserID|    AverageAmount|
+-------+-----------------+
|U591589|369.2982236532613|
|U315078|492.9267545184986|
|U038474|653.3242114636305|
|U238706|615.5161955073115|
|U861987|545.4259967251896|
+-------+-----------------+
only showing top 5 rows



In [24]:
# (4) Identify the top 10 users who have the highest average 
#     transaction amounts.
(df_sales.groupBy('UserID')
         .agg(F.avg('Amount').alias('AverageAmount'))
         .orderBy(F.col('AverageAmount').desc())).show(10)

+-------+-----------------+
| UserID|    AverageAmount|
+-------+-----------------+
|U690650|999.2198137461229|
|U883885|999.0292010956597|
|U245368|994.4741714498298|
|U214052|989.0303576861718|
|U965530|986.9511285805534|
|U691660|985.7412265265561|
|U218142|983.3017817031161|
|U437800|979.7634321093049|
|U695841|978.7352767314061|
|U011250|977.6680968053065|
+-------+-----------------+
only showing top 10 rows



In [25]:
(df_sales.groupBy('UserID')
         .agg(F.avg('Amount').alias('AverageAmount'))
         .orderBy(F.desc('AverageAmount'))).show(10)

+-------+-----------------+
| UserID|    AverageAmount|
+-------+-----------------+
|U690650|999.2198137461229|
|U883885|999.0292010956597|
|U245368|994.4741714498298|
|U214052|989.0303576861718|
|U965530|986.9511285805534|
|U691660|985.7412265265561|
|U218142|983.3017817031161|
|U437800|979.7634321093049|
|U695841|978.7352767314061|
|U011250|977.6680968053065|
+-------+-----------------+
only showing top 10 rows



-------------------------------------